In [0]:
%pip install python-louvain networkx pandas pyarrow

import networkx as nx
import community as community_louvain  
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from datetime import datetime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/204.6 kB ? eta -:--:--
     ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/204.6 kB 1.2 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 92.2/204.6 kB 1.4 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency information for networkx from https://files.pythonhosted.org/packages/9e/c9/b2622292ea83fbb4ec318f5b9ab867d0a28ab43c5717bb85b0a5f6b3b0a4/networkx-3.6.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.1 MB ? eta -:--:--
   ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.2/2.1 MB 13.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━ 0.7/2.1 MB 12.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 1.7/2.1 MB 17.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.2 MB/s eta 0:00:00
  Cr

In [0]:
import pyspark.sql.functions as F

storage_account = "chungminhlamkpdl"
container = "reddit-data"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    "yoer_key" 
)

sim_path = f"wasbs://{container}@{storage_account}.blob.core.windows.net/subreddit_similarity_results.csv"

df_sim = spark.read.option("header", True).csv(sim_path)
df_sim = df_sim.withColumn("Similarity_Score", F.col("Similarity_Score").cast("float"))

print(f"Tổng số cặp similarity: {df_sim.count()}")

Tổng số cặp similarity: 3225710


In [0]:
threshold_row = df_sim.approxQuantile("Similarity_Score", [0.97], 0.001)
threshold = threshold_row[0]
print(f"Threshold (97th percentile): {threshold:.4f}")

df_filtered = df_sim.filter(F.col("Similarity_Score") >= threshold)
print(f"Số cạnh sau lọc: {df_filtered.count()}")

Threshold (97th percentile): 0.7343
Số cạnh sau lọc: 99014


In [0]:
edges_pd = df_filtered.select("Subreddit_A", "Subreddit_B", "Similarity_Score").toPandas()

In [0]:
G = nx.Graph()
for _, row in edges_pd.iterrows():
    G.add_edge(
        row["Subreddit_A"],
        row["Subreddit_B"],
        weight=float(row["Similarity_Score"])
    )

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph: 3256 nodes, 99014 edges


In [0]:
partition = community_louvain.best_partition(G, weight='weight', resolution=1.0)

num_communities = len(set(partition.values()))
print(f"Số communities phát hiện được: {num_communities}")

Số communities phát hiện được: 91


In [0]:
modularity = community_louvain.modularity(partition, G, weight='weight')
print(f"Modularity score: {modularity:.4f}")

Modularity score: 0.6293


In [0]:
community_df = pd.DataFrame([
    {"subreddit": sub, "community_id": comm_id}
    for sub, comm_id in partition.items()
])

community_sizes = community_df.groupby("community_id").size().reset_index(name="community_size")
community_df = community_df.merge(community_sizes, on="community_id")
community_df = community_df.sort_values("community_size", ascending=False)

print("\nTop 10 communities lớn nhất:")
print(community_df.groupby("community_id")["community_size"].first()
      .sort_values(ascending=False).head(10))


Top 10 communities lớn nhất:
community_id
0     605
29    373
9     331
50    280
24    228
46    195
75    190
21    164
7     163
3     106
Name: community_size, dtype: int64


In [0]:
df_spark = spark.createDataFrame(community_df)
out_path = f"wasbs://{container}@{storage_account}.blob.core.windows.net/community_results"
df_spark.coalesce(1).write.mode("overwrite").option("header", True).csv(out_path)
print(f"Đã lưu community results")

Đã lưu community results
